# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaGaya/KGaya/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My Rule: Stale Visible Page with Low CTR

A page is flagged for review if it:
- Has not been updated in 180+ days (staleness signal)
- Still receives 100+ impressions (visible enough to matter)
- Has a CTR below 0.5% (under-capturing clicks for its visibility)

The score combines these three signals into a weighted refresh priority score.

Reason codes this rule can output:
- stale_visible_page: days_since_last_update >= 180 AND impressions_90d >= 100
- low_ctr_visible_page: impressions_90d >= 500 AND ctr < 0.005 AND avg_position <= 20
- stale_and_low_ctr: both conditions met (highest priority)

Action labels:
- REVIEW_CONTENT: page is stale and visible — update the content
- REVIEW_METADATA: page has impressions but low CTR — fix title/meta description
- REVIEW_BOTH: page meets both conditions — content and metadata review needed

In [9]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                        REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df)} rows")

# Create label from trend_direction
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print(f"Declining pages: {df['is_declining_label'].mean():.1%}")

# SIGNAL CHECK 1 — Staleness vs decline rate
staleness_buckets = pd.cut(df['days_since_last_update'],
    bins=[0, 90, 180, 365, 999999],
    labels=['0-90 days', '91-180 days', '181-365 days', '365+ days'])

sig1 = df.groupby(staleness_buckets, observed=True)['is_declining_label'].agg(
    n='count', decline_rate='mean').reset_index()
sig1['decline_rate'] = sig1['decline_rate'].round(3)
print("\nSignal 1 — Staleness vs Decline Rate:")
print(sig1.to_string(index=False))
print("\nVerdict: CONFIRMED or MIXED — check your output")

# SIGNAL CHECK 2 — CTR buckets vs decline rate
ctr_buckets = pd.cut(df['ctr'],
    bins=[-0.001, 0.01, 0.05, 0.15, 1.0],
    labels=['Very Low (<1%)', 'Low (1-5%)', 'Medium (5-15%)', 'High (>15%)'])

sig2 = df.groupby(ctr_buckets, observed=True)['is_declining_label'].agg(
    n='count', decline_rate='mean').reset_index()
sig2['decline_rate'] = sig2['decline_rate'].round(3)
print("\nSignal 2 — CTR vs Decline Rate:")
print(sig2.to_string(index=False))
print("\nVerdict: CONFIRMED or MIXED — check your output")

Loaded 30000 rows
Declining pages: 54.2%

Signal 1 — Staleness vs Decline Rate:
days_since_last_update     n  decline_rate
             0-90 days 20655         0.512
           91-180 days  9171         0.611
          181-365 days   169         0.467
             365+ days     5         0.600

Verdict: CONFIRMED or MIXED — check your output

Signal 2 — CTR vs Decline Rate:
           ctr     n  decline_rate
Very Low (<1%) 13290         0.498
    Low (1-5%)  1129         0.717
Medium (5-15%)  4103         0.641
   High (>15%)  9789         0.557

Verdict: CONFIRMED or MIXED — check your output


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Building the ranked queue using the three signals above.
Score is a weighted combination of staleness, visibility and CTR gap.
Higher score = higher review priority.
Written to work/outputs/baseline_action_score.csv.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# =====================
# FEATURE ENGINEERING
# =====================

# Normalise signals to 0-1 scale
df['staleness_score'] = np.clip(df['days_since_last_update'] / 365, 0, 1)
df['visibility_score'] = np.clip(np.log1p(df['impressions_90d']) / np.log1p(df['impressions_90d'].max()), 0, 1)
df['ctr_gap_score'] = np.clip(1 - (df['ctr'] / 0.05), 0, 1)  # gap from 5% benchmark

# =====================
# BASELINE SCORE
# =====================
# Weights: staleness 40%, visibility 35%, CTR gap 25%
df['baseline_score'] = (
    0.40 * df['staleness_score'] +
    0.35 * df['visibility_score'] +
    0.25 * df['ctr_gap_score']
)

# =====================
# REASON CODES
# =====================
def assign_reason(row):
    stale = row['days_since_last_update'] >= 180
    visible = row['impressions_90d'] >= 100
    low_ctr = row['ctr'] < 0.005 and row['impressions_90d'] >= 500

    if stale and visible and low_ctr:
        return 'stale_and_low_ctr'
    elif stale and visible:
        return 'stale_visible_page'
    elif low_ctr:
        return 'low_ctr_visible_page'
    else:
        return 'monitor'

def assign_action(reason):
    if reason == 'stale_and_low_ctr':
        return 'REVIEW_BOTH'
    elif reason == 'stale_visible_page':
        return 'REVIEW_CONTENT'
    elif reason == 'low_ctr_visible_page':
        return 'REVIEW_METADATA'
    else:
        return 'MONITOR'

df['reason_code'] = df.apply(assign_reason, axis=1)
df['action_label'] = df['reason_code'].apply(assign_action)

# =====================
# RANKED QUEUE
# =====================
queue = df[[
    'content_id', 'impressions_90d', 'days_since_last_update',
    'ctr', 'avg_position', 'trend_direction',
    'baseline_score', 'reason_code', 'action_label'
]].sort_values('baseline_score', ascending=False).reset_index(drop=True)

queue['rank'] = queue.index + 1

# =====================
# WRITE CSV
# =====================
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Queue written: {len(queue)} rows")
print(f"\nReason code distribution:")
print(queue['reason_code'].value_counts())
print(f"\nAction label distribution:")
print(queue['action_label'].value_counts())
print(f"\nTop 5 preview:")
print(queue.head())

Queue written: 30000 rows

Reason code distribution:
reason_code
monitor                 27429
low_ctr_visible_page     2536
stale_visible_page         33
stale_and_low_ctr           2
Name: count, dtype: int64

Action label distribution:
action_label
MONITOR            27429
REVIEW_METADATA     2536
REVIEW_CONTENT        33
REVIEW_BOTH            2
Name: count, dtype: int64

Top 5 preview:
             content_id  impressions_90d  days_since_last_update  ctr  \
0  content_55a5b1c46474               35                     373  0.0   
1  content_6476d1d8c050              304                     313  0.0   
2  content_02b0d6e30129              176                     313  0.0   
3  content_d25a099b3726              202                     305  0.0   
4  content_f488400fca67              155                     305  0.0   

   avg_position trend_direction  baseline_score         reason_code  \
0           7.5            down        0.745327             monitor   
1          67.8          

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-20 review — one line per row: action, reason code, confidence,
and what would make it wrong.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Print top 20 for review
top20 = queue.head(20)
print("TOP 20 REVIEW")
print("="*80)
for _, row in top20.iterrows():
    print(f"\nRank {int(row['rank'])}: {row['action_label']} | {row['reason_code']}")
    print(f"  Impressions: {row['impressions_90d']:.0f} | Days stale: {row['days_since_last_update']:.0f} | CTR: {row['ctr']:.4f} | Position: {row['avg_position']:.1f} | Trend: {row['trend_direction']}")

# Precision@20 — how many top 20 are actually declining?
top20_precision = (top20['trend_direction'] == 'down').mean()
print(f"\nPrecision@20: {top20_precision:.3f} ({int(top20_precision*20)} of top 20 are actually declining)")

TOP 20 REVIEW

Rank 1: MONITOR | monitor
  Impressions: 35 | Days stale: 373 | CTR: 0.0000 | Position: 7.5 | Trend: down

Rank 2: REVIEW_CONTENT | stale_visible_page
  Impressions: 304 | Days stale: 313 | CTR: 0.0000 | Position: 67.8 | Trend: up

Rank 3: REVIEW_CONTENT | stale_visible_page
  Impressions: 176 | Days stale: 313 | CTR: 0.0000 | Position: 6.9 | Trend: down

Rank 4: REVIEW_CONTENT | stale_visible_page
  Impressions: 202 | Days stale: 305 | CTR: 0.0000 | Position: 64.5 | Trend: up

Rank 5: REVIEW_CONTENT | stale_visible_page
  Impressions: 155 | Days stale: 305 | CTR: 0.0000 | Position: 5.7 | Trend: down

Rank 6: MONITOR | monitor
  Impressions: 95 | Days stale: 313 | CTR: 0.0000 | Position: 67.6 | Trend: down

Rank 7: MONITOR | monitor
  Impressions: 30 | Days stale: 334 | CTR: 0.0000 | Position: 9.3 | Trend: down

Rank 8: REVIEW_CONTENT | stale_visible_page
  Impressions: 103 | Days stale: 304 | CTR: 0.0000 | Position: 8.9 | Trend: stable

Rank 9: MONITOR | monitor
  Impre

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks and leakage check.

Weak picks: pages flagged REVIEW_CONTENT purely because they are old
but have stable or growing trend direction. Age alone is not enough —
a page that is old but performing well does not need a content review.
The staleness signal may be too aggressive without a trend filter.

Pages with avg_position > 50 that appear in the top 20 are also
suspect — they have very low visibility and low CTR almost by default,
not because of a content problem.

Leakage check:
- No product flags used (health_score, priority_score not in dataset)
- No future-window data used — all signals are from the observable 90-day window
- trend_direction used only as a label for Precision@K evaluation,
  never as a feature in the score
- is_declining_label not used as a feature anywhere in the scoring formula

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak picks — flagged but not actually declining
weak = top20[top20['trend_direction'] != 'down']
print(f"Weak picks in top 20 (not declining): {len(weak)}")
print(weak[['rank', 'reason_code', 'action_label', 'days_since_last_update',
            'impressions_90d', 'ctr', 'trend_direction']].to_string(index=False))

# Leakage check — confirm no label-derived columns in score
print("\nLeakage check:")
print("Features used in baseline_score: staleness_score, visibility_score, ctr_gap_score")
print("trend_direction used: ONLY for Precision@K evaluation (not a feature)")
print("is_declining_label used: ONLY for Precision@K evaluation (not a feature)")
print("Product flags used: NONE (not in dataset)")
print("Future window data used: NONE")
print("LEAKAGE STATUS: CLEAN")

# Baseline metrics summary
from sklearn.metrics import average_precision_score
y_true = (queue['trend_direction'] == 'down').astype(int)
precision_at_20 = (queue.head(20)['trend_direction'] == 'down').mean()
precision_at_50 = (queue.head(50)['trend_direction'] == 'down').mean()
print(f"\nBaseline Metrics:")
print(f"Precision@20: {precision_at_20:.3f}")
print(f"Precision@50: {precision_at_50:.3f}")
print(f"This is the score the Week 5 model must beat.")

Weak picks in top 20 (not declining): 8
 rank        reason_code   action_label  days_since_last_update  impressions_90d  ctr trend_direction
    2 stale_visible_page REVIEW_CONTENT                     313              304  0.0              up
    4 stale_visible_page REVIEW_CONTENT                     305              202  0.0              up
    8 stale_visible_page REVIEW_CONTENT                     304              103  0.0          stable
   10            monitor        MONITOR                     301               64  0.0              up
   13            monitor        MONITOR                     334               10  0.0            flat
   16            monitor        MONITOR                     372                1  0.0             new
   18            monitor        MONITOR                     305               17  0.0              up
   20            monitor        MONITOR                     305               15  0.0              up

Leakage check:
Features used in baseline_

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.